In [ ]:
# 01 — Build STOXX 600 Daily Returns Panel

**Purpose**: Construct a survivorship-bias-free daily total return panel for the 
historical STOXX Europe 600 universe over 2013-2025.

**Inputs**:
- `data/factset_stoxx600.xlsx` — FactSet current constituents (validation)
- `data/compustat_constituents.xlsx` — Compustat GVKEYX 150376 history
- WRDS access to `comp.g_secd`, `comp.g_security`, `comp.g_company`

**Output**: `data python/stoxx600_daily_returns_2013_2025.parquet`

**Run time**: ~5 minutes (WRDS pull dominates)

See `README.md` in project root for full pipeline documentation.

In [ ]:
import pandas as pd


# --- Load both files ---
factset = pd.read_excel('/Users/andradaduta/Desktop/Master Thesis/Python Data/Stoxx/factset_stoxx600.xlsx')
compustat = pd.read_excel('/Users/andradaduta/Desktop/Master Thesis/Python Data/Stoxx/compustat_stoxx600.xlsx')



# Check columns so we know the exact ISIN field names
print("FactSet columns:", factset.columns.tolist())
print("Compustat columns:", compustat.columns.tolist())

In [ ]:
# COmpustat vs Factset Stoxx Validity check 
#--- Adjust these to match your actual column names ---
FACTSET_ISIN = 'Unnamed: 4'
COMPUSTAT_ISIN = '(isin) International Security Identification Number'
COMPUSTAT_THRU = '(thru) Effective Thru Date'   # the Effective Thru date column

# --- Build the two sets (Compustat: only firms active in 2025) ---
factset_isins = set(factset[FACTSET_ISIN].dropna().str.strip().str.upper())

compustat_2025 = compustat[compustat[COMPUSTAT_THRU].isna()]
compustat_isins = set(compustat_2025[COMPUSTAT_ISIN].dropna().str.strip().str.upper())

# --- Compare ---
overlap = factset_isins & compustat_isins
print(f"FactSet:   {len(factset_isins)}")
print(f"Compustat: {len(compustat_isins)}")
print(f"Overlap:   {len(overlap)} ({len(overlap)/len(factset_isins):.1%})")
print(f"FactSet only:   {len(factset_isins - compustat_isins)}")
print(f"Compustat only: {len(compustat_isins - factset_isins)}")

In [ ]:
factset['Unnamed: 4'].head(10)

In [ ]:
!pip install wrds

In [ ]:
import pandas as pd
import numpy as np
import wrds

db = wrds.Connection(wrds_username='your_wrds_username')  # replace with your username

In [ ]:
# Create Universe 

COMPUSTAT_FROM = 'from'   # adjust

universe = compustat[COMPUSTAT_ISIN].dropna().str.strip().str.upper().unique().tolist()
print(f"Universe size: {len(universe)} unique ISINs")

In [ ]:
#SQL data pull 
query = """
    SELECT gvkey, iid, datadate, isin,
           prccd, ajexdi, trfd, cshoc, curcdd, exchg, fic
    FROM comp.g_secd
    WHERE isin IN %(isins)s
      AND datadate BETWEEN '2013-01-01' AND '2025-12-31'
"""

prices = db.raw_sql(query, params={'isins': isin_tuple})
print(f"Pulled {len(prices):,} rows for {prices['isin'].nunique()} unique securities")
prices.head()

In [ ]:
#checking other variables / column to add from SQL
# See what columns g_security actually has
columns = db.raw_sql("""
    SELECT column_name 
    FROM information_schema.columns 
    WHERE table_schema = 'comp' AND table_name = 'g_security'
    ORDER BY column_name
""")
print(columns.to_string())

In [ ]:
static_query = """
    SELECT gvkey, iid, tic, isin, sedol, cusip, 
           exchg, excntry, tpci, secstat, dlrsni, dldtei
    FROM comp.g_security
    WHERE isin IN %(isins)s
"""
static = db.raw_sql(static_query, params={'isins': isin_tuple})
print(f"Static info: {len(static)} rows")
static.head()

In [ ]:
# Get unique gvkeys from your static dataframe
gvkeys = tuple(static['gvkey'].dropna().unique())

names_query = """
    SELECT gvkey, conm, conml, fic, gsector, ggroup, gind, gsubind
    FROM comp.g_company
    WHERE gvkey IN %(gvkeys)s
"""
try:
    names = db.raw_sql(names_query, params={'gvkeys': gvkeys})
    print(f"Names found: {len(names)} firms")
    names.head()
except Exception as e:
    print(f"g_company didn't work: {e}")
    # Try alternative
    names = db.raw_sql("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_schema = 'comp' AND table_name = 'g_company'
        ORDER BY column_name
    """)
    print("Available columns in g_company:")
    print(names.to_string())

In [ ]:
# Merge static identifiers with company info
firm_info = static.merge(names, on='gvkey', how='left')

# Then merge into the prices panel
prices = prices.merge(firm_info, on=['gvkey', 'iid'], how='left', suffixes=('', '_dup'))

# Drop any duplicated columns from the merge
prices = prices[[c for c in prices.columns if not c.endswith('_dup')]]

print(f"Final prices: {len(prices):,} rows")
print(f"Columns: {prices.columns.tolist()}")
prices[['datadate', 'isin', 'tic', 'conm', 'gsector', 'prccd']].head(10)

In [ ]:
#Sanity check for the panel data 

In [ ]:
# 1. Sector distribution — should look like a diversified European index
print(prices.drop_duplicates('gvkey')['gsector'].value_counts())

In [ ]:
# 2. Country distribution — listing country
print(prices.drop_duplicates('gvkey')['excntry'].value_counts().head(15))

In [ ]:
# 3. Currency distribution
print(prices['curcdd'].value_counts().head(10))

In [ ]:
# 4. Spot-check a known firm
print(prices[prices['conm'].str.contains('SHELL', case=False, na=False)][
    ['datadate', 'conm', 'tic', 'prccd', 'curcdd', 'gsector']
].head(10))

In [ ]:
# Pick a few well-known names you can verify
for name in ['NESTLE', 'NOVARTIS', 'ASML', 'BP', 'TOTAL', 'BMW', 'SIEMENS', 'LVMH', 'SAP']:
    mask = prices['conml'].str.contains(name, case=False, na=False)
    found = prices[mask][['conm', 'conml', 'tic', 'isin', 'gsector']].drop_duplicates()
    if len(found) > 0:
        print(f"\n{name}:")
        print(found.head(3))

In [ ]:
#Doing Returns
# Sort so shift() works correctly within firm
prices = prices.sort_values(['gvkey', 'iid', 'datadate']).copy()
prices['datadate'] = pd.to_datetime(prices['datadate'])

# Total return index: adjusts for both splits and dividends
prices['tri'] = prices['prccd'] / prices['ajexdi'] * prices['trfd']

# Daily log return, computed within each firm-share-class
prices['ret'] = (
    prices.groupby(['gvkey', 'iid'])['tri']
          .transform(lambda x: np.log(x / x.shift(1)))
)

# Drop the first observation per firm (no prior price to compute return from)
prices = prices.dropna(subset=['ret'])
print(f"Returns computed: {len(prices):,} firm-day observations")
prices[['datadate', 'conm', 'prccd', 'tri', 'ret']].head(10)

In [ ]:
# Distribution of daily returns
print(prices['ret'].describe())

# Should look like: mean ~0.0003 (≈8% annual), std ~0.015-0.020 (≈25-30% annual), 
# min around -0.3 (max one-day crash), max around +0.3

In [ ]:
#Min / Max Issue Diagnose 
# Look at the worst offenders
extreme = prices[prices['ret'].abs() > 0.5].sort_values('ret', key=abs, ascending=False)
print(f"Returns > 50% in absolute value: {len(extreme)}")
print(extreme[['datadate', 'conm', 'isin', 'prccd', 'ajexdi', 'trfd', 'tri', 'ret']].head(30))


In [ ]:
#Min/Max Diagnosis Issue
#See if the worst offenders have consistent data issues
extreme_firm_counts = prices[prices['ret'].abs() > 1.0]['conm'].value_counts()
print(extreme_firm_counts.head(20))
#Shows no data issues, just extreme events

In [ ]:
#Decided to drop the extrme days that are due to splits 
import numpy as np
from scipy.stats import mstats

# Step 1: Hard filter — drop pure adjustment artefacts
n_before = len(prices)
returns_panel = prices[prices['ret'].abs() < 1.0].copy()
print(f"Dropped {n_before - len(returns_panel)} artefact rows ({(n_before - len(returns_panel))/n_before:.4%})")
print(f"Panel size: {len(returns_panel):,}")

# Step 2: Winsorize at 0.1% / 99.9% for the analysis version of returns
lower = returns_panel['ret'].quantile(0.001)
upper = returns_panel['ret'].quantile(0.999)
returns_panel['ret_w'] = returns_panel['ret'].clip(lower, upper)
print(f"\nWinsorization bounds: [{lower:.4f}, {upper:.4f}]")

# Final summary
print("\nRaw returns:")
print(returns_panel['ret'].describe())
print("\nWinsorized returns:")
print(returns_panel['ret_w'].describe())

In [ ]:
# save the panel data to parquet since is fastest to load 
returns_panel.to_parquet('stoxx600_daily_returns_2013_2025.parquet')
print(f"Saved: {len(returns_panel):,} rows, {returns_panel['gvkey'].nunique()} firms")

In [ ]:
panel = pd.read_parquet('stoxx600_daily_returns_2013_2025.parquet')